# 07. Cost와 Latency 설계 비교

- reactive routing과 static routing 비교
- cold start와 자원 비용 trade-off 이해

## Cost-optimized router

동적으로 학습한 warm backend map을 사용한다.

In [ ]:
"""Cost-optimized routing layer (Figure 3-11, part A).

Keeps a model -> backend mapping so a request is sent to an instance that
already has the model loaded, and falls back to the least loaded instance.
The mapping is refreshed from each backend's /models endpoint, which is why
this design is reactive: it only learns about placement after the fact.
"""

import asyncio
import os
import time
from collections import defaultdict
from typing import Any

import httpx
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict

BACKENDS: list[str] = [b for b in os.getenv("BACKENDS", "").split(",") if b]
REFRESH_SECONDS = float(os.getenv("REFRESH_SECONDS", "2"))

app = FastAPI(title="ch03 cost-optimized router")
routing_map: dict[str, list[str]] = defaultdict(list)
inflight: dict[str, int] = defaultdict(int)
metrics = {"sticky_hits": 0, "fallbacks": 0}


class PredictionRequest(BaseModel):
  model_config = ConfigDict(protected_namespaces=())
  model_id: str
  input_data: Any


async def refresh_routing_map():
  async with httpx.AsyncClient(timeout=5) as client:
    while True:
      updated: dict[str, list[str]] = defaultdict(list)
      for backend in BACKENDS:
        try:
          response = await client.get(f"{backend}/models")
          for model_id in response.json().get("loaded_models", {}):
            updated[model_id].append(backend)
        except Exception:
          continue
      routing_map.clear()
      routing_map.update(updated)
      await asyncio.sleep(REFRESH_SECONDS)


def pick_backend(model_id: str) -> str:
  warm = routing_map.get(model_id) or []
  if warm:
    metrics["sticky_hits"] += 1
    return min(warm, key=lambda b: inflight[b])
  if not BACKENDS:
    raise HTTPException(status_code=503, detail="no backends configured")
  metrics["fallbacks"] += 1
  return min(BACKENDS, key=lambda b: inflight[b])


@app.on_event("startup")
async def startup():
  asyncio.create_task(refresh_routing_map())


@app.post("/predict")
async def predict(request: PredictionRequest):
  backend = pick_backend(request.model_id)
  inflight[backend] += 1
  started = time.perf_counter()
  try:
    async with httpx.AsyncClient(timeout=300) as client:
      response = await client.post(f"{backend}/predict", json=request.model_dump())
      response.raise_for_status()
      body = response.json()
  finally:
    inflight[backend] -= 1

  body["_routing"] = {
    "backend": backend,
    "warm": backend in (routing_map.get(request.model_id) or []),
    "router_seconds": round(time.perf_counter() - started, 4),
  }
  return body


@app.get("/routing_map")
async def get_routing_map():
  return {"backends": BACKENDS, "map": routing_map, "inflight": inflight, **metrics}


@app.get("/healthz")
async def healthz():
  return {"status": "ok", "backends": len(BACKENDS)}

## warm backend 선택 직접 실행

라우팅 map과 round-robin cursor를 직접 바꿔 결과를 확인한다.

In [ ]:
routing_map.clear()
routing_map["sentiment"] = ["http://worker-a", "http://worker-b"]
[pick_backend("sentiment") for _ in range(4)]

## Latency-optimized router

사전에 배치한 model-to-backend map을 사용한다.

In [ ]:
import os
from pathlib import Path

routing_path = Path("routing.json")
if not routing_path.exists():
  routing_path = Path("07_tradeoff/routing.json")
os.environ["ROUTING_MAP_PATH"] = str(routing_path)

In [ ]:
"""Latency-optimized routing layer (Figure 3-12, part A).

The map is static and comes from the provisioning step, not from observing
traffic. Every model already has a dedicated, always-on service group, so the
router only has to look the model up and forward.
"""

import json
import os
import time
from typing import Any

import httpx
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict

ROUTING_MAP_PATH = os.getenv("ROUTING_MAP_PATH", "07_tradeoff/routing.json")

app = FastAPI(title="ch03 latency-optimized router")

with open(ROUTING_MAP_PATH) as f:
  ROUTING_MAP: dict[str, str] = json.load(f)


class PredictionRequest(BaseModel):
  model_config = ConfigDict(protected_namespaces=())
  model_id: str
  input_data: Any


@app.post("/predict")
async def predict(request: PredictionRequest):
  target = ROUTING_MAP.get(request.model_id)
  if target is None:
    raise HTTPException(status_code=404, detail=f"model {request.model_id} is not provisioned")

  started = time.perf_counter()
  async with httpx.AsyncClient(timeout=300) as client:
    response = await client.post(f"{target}/predict", json=request.model_dump())
    response.raise_for_status()
    body = response.json()

  body["_routing"] = {
    "backend": target,
    "warm": True,
    "router_seconds": round(time.perf_counter() - started, 4),
  }
  return body


@app.get("/routing_map")
async def get_routing_map():
  return {"map": ROUTING_MAP}


@app.get("/healthz")
async def healthz():
  return {"status": "ok", "provisioned_models": len(ROUTING_MAP)}

## 퀴즈

코드를 다시 보지 않고 먼저 답해본다.

1. cost-optimized 설계에서 p50보다 p95와 max latency가 크게 나타나는 원인은 무엇인가?
2. latency-optimized 설계에서 model 수가 늘면 어떤 자원이 선형 증가하는가?

<details>
<summary>정답과 해설 보기</summary>

1. cache miss 요청에만 model load cold start가 추가되기 때문이다. 대부분의 warm 요청은 빠르지만 일부 miss가 tail latency를 크게 만든다.
2. model별 전용 Deployment·Pod와 사전 로드된 CPU·GPU·메모리 용량이 model 수에 따라 증가한다.

</details>